## Notebook for SpeakEasyR testing

- In this notebook we are going to run the SpeakEasy algorithm on cross region adjecency matrix of ROSMAP data from 3 regions,AC,PCGBA23, MFBA9BA46.

### We start by loading our adj matrix that normalized with cross tissue beta of 2 and tissue specific beta of 3.

In [1]:
# Load .env variables
library(dotenv)
devtools::load_all()
load_dot_env(file = ".env")

# Access environment variables
ADJMAT <- Sys.getenv("ADJMATRIXCT2TS3")

ℹ Loading speakeasyR


In [2]:
ADJMAT

[1] "/media/psylab-6028/DATA/Eden/CoExpression_ReProduction/nbs/xwgcna_rosmap_constBeta_CT2_TS3_adjacency.csv"

In [3]:
file.info(ADJMAT)$size

[1] 53608364547

- The adj matrix size is ~53 Gb, so we load it with fread.

In [4]:
library(data.table)

In [5]:
adj <- fread(ADJMAT, header = TRUE)
adj <- as.data.frame(adj)
rownames(adj) <- adj[[1]]
adj[[1]] <- NULL

In [6]:
dim(adj)

[1] 51918 51918

- The dimensions of our adj matrix is 51,918 rows x 51,918 columns (17,306 genes from 3 tissues).
- We convert the adj data frame into matrix for the speakEasy algorithm.

In [8]:
adj_matrix <- as.matrix(adj)

In [12]:
adj_matrix[1:5, 1:5]

,AC_ENSG00000000003.14,AC_ENSG00000000419.12,AC_ENSG00000000457.13,AC_ENSG00000000460.16,AC_ENSG00000000938.12
AC_ENSG00000000003.14,0.000000e+00,1.148588e-02,4.194853e-04,9.910762e-06,5.534821e-04
AC_ENSG00000000419.12,1.148588e-02,0.000000e+00,6.550656e-03,1.395688e-05,8.605881e-05
AC_ENSG00000000457.13,4.194853e-04,6.550656e-03,0.000000e+00,5.403395e-03,5.086677e-07
AC_ENSG00000000460.16,9.910762e-06,1.395688e-05,5.403395e-03,0.000000e+00,5.851949e-05
AC_ENSG00000000938.12,5.534821e-04,8.605881e-05,5.086677e-07,5.851949e-05,0.000000e+00


- Importing the SpeakEasyR.

In [9]:
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/cluster_genes.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/cluster.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/knn_graph.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/order_nodes.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/speakeasyR-package.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/utils.R")

- Assiging the parameters for the cluster function that get the adj matrix.

In [11]:
is_directed <- FALSE
discard_transient <- 3
independent_runs <- 10
max_threads <- 0
seed <- 123
target_clusters <- 0
target_partitions <- 20
subcluster <- 3
min_clust <- 30
verbose <- FALSE

In [13]:
results <- speakeasyR::cluster(adj_matrix,
    discard_transient = discard_transient,
    independent_runs = independent_runs, max_threads = max_threads, seed = seed,
    target_clusters = target_clusters, target_partitions = target_partitions,
    subcluster = subcluster, min_clust = min_clust, verbose = verbose,
    is_directed = is_directed
  )

ERROR: Error in speakeasyR::cluster(adj_matrix, discard_transient = discard_transient, : long vectors (argument 3) are not supported in .C


In [17]:
print_speakeasy_env_snapshot <- function(adj_matrix = NULL,
                                         extra_pkgs = c("speakeasyR","WGCNA","Matrix")) {
  sep <- function(title) {
    cat("\n", paste0(rep("=", nchar(title)+4), collapse=""), "\n",
        "= ", title, " =\n",
        paste0(rep("=", nchar(title)+4), collapse=""), "\n", sep="")
  }
  sh <- function(cmd) {
    out <- tryCatch(system(cmd, intern = TRUE, ignore.stderr = TRUE), error = function(e) character())
    if (length(out)) cat("$", cmd, "\n", paste(out, collapse="\n"), "\n", sep="") else {
      cat("$", cmd, "\n", "(no output / command not available)\n", sep="")
    }
  }
  kv <- function(k, v) cat(sprintf("  %-28s : %s\n", k, v))

  # --- Header
  sep("SPEAKEASY2 – System & Runtime Environment (print)")
  kv("Generated", format(Sys.time(), "%Y-%m-%d %H:%M:%S %Z"))
  info <- Sys.info()  # Linux
  kv("OS", paste(info["sysname"], info["release"], info["version"]))
  kv("Machine", paste(info["machine"]))
  kv("R version", paste(R.version$major, R.version$minor, sep="."))

  sep("Operating System & Kernel")
  sh("uname -a")
  sh("bash -lc 'cat /etc/os-release 2>/dev/null || true'")
  sh("bash -lc 'lsb_release -a 2>/dev/null || true'") 

  sep("CPU")
  sh("bash -lc 'lscpu 2>/dev/null || true'")         
  sh("bash -lc \"grep -m1 'model name' /proc/cpuinfo 2>/dev/null || true\"")
  kv("Logical cores (R)", parallel::detectCores())

  sep("Memory & Storage")
  sh("bash -lc 'free -h 2>/dev/null || true'")         
  sh("bash -lc 'df -h 2>/dev/null || true'")

  sep("GPU / Graphics")
  sh("bash -lc \"lspci | grep -i -E 'vga|3d|nvidia|amd' 2>/dev/null || true\"")

  sep("R Session Info")
  print(sessionInfo())

  sep("R Build / Toolchain / BLAS-LAPACK")
  print(R.version)
  print(extSoftVersion())
  sh("R CMD config CC")
  sh("R CMD config CFLAGS")
  sh("R CMD config CXX")
  sh("R CMD config CPPFLAGS")
  sh("R CMD config LDFLAGS")
  sh("bash -lc 'ldconfig -p 2>/dev/null | grep -Ei \"blas|lapack\" || true'")
  sh("update-alternatives --display libblas 2>/dev/null") 

  sep("R Machine Limits & Capabilities")
  kv(".Machine$sizeof.pointer", .Machine$sizeof.pointer)
  kv(".Machine$integer.max", format(.Machine$integer.max, big.mark=","))
  caps <- names(which(capabilities()))
  kv("capabilities()", if (length(caps)) paste(caps, collapse=", ") else "<none>")
  sh("ulimit -a")
  sh("getconf PAGE_SIZE")
  sh("getconf LONG_BIT")

  sep("Selected Packages")
  ip <- utils::installed.packages()
  keep <- rownames(ip)[rownames(ip) %in% unique(c(extra_pkgs, "speakeasyR"))]
  if (length(keep)) {
    df <- as.data.frame(ip[keep, c("Package","Version","LibPath","Built"), drop = FALSE], stringsAsFactors = FALSE)
    print(df, row.names = FALSE)
  } else {
    cat("(Selected packages not found)\n")
  }

  sep("Workload Object (adj_matrix) Summary")
  if (!is.null(adj_matrix)) {
    dims <- tryCatch(dim(adj_matrix), error = function(e) NULL)
    len  <- tryCatch(length(adj_matrix), error = function(e) NA_integer_)
    bytes <- tryCatch(as.numeric(object.size(adj_matrix)), error = function(e) NA_real_)
    if (!is.null(dims)) kv("dim", paste(dims, collapse=" x "))
    kv("elements (length)", format(len, big.mark = ","))
    kv("object.size (GB)", if (is.finite(bytes)) sprintf("%.2f", bytes/1024^3) else "NA")
    thr <- .Machine$integer.max
    kv("2^31 - 1 threshold", format(thr, big.mark = ","))
    kv("Exceeds 2^31-1?", ifelse(is.finite(len) && len > thr, "YES", "NO"))
    if (!is.null(dims) && length(dims) == 2) {
      kv("Square-equivalent n (if n x n)", ifelse(dims[1] == dims[2], dims[1], "not square"))
    }
  } else {
    cat("Set adj_matrix= to include size diagnostics (dim/length/object.size vs 2^31-1).\n")
  }

  sep("Helpers: long-vector calculators")
  cat("check_matrix_limit(n, m = n): returns list with elements, exceeds_2^31_1\n")
  check_matrix_limit <- function(n, m = n) {
    elems <- as.double(n) * as.double(m)
    list(n = n, m = m, elements = elems,
         exceeds_2_31_1 = elems > .Machine$integer.max)
  }
  print(check_matrix_limit)
  assign("check_matrix_limit", check_matrix_limit, envir = .GlobalEnv)

  cat("\nNote on `.C` long vectors:\n",
      "  `.C` does NOT support vectors with length > 2^31-1. ",
      "Support usually requires `.Call` and `R_xlen_t`, or chunked/sparse designs.\n", sep="")
}


In [18]:
print_speakeasy_env_snapshot(adj_matrix = adj_matrix)


= SPEAKEASY2 – System & Runtime Environment (print) =
  Generated                    : 2025-09-22 10:35:33 IDT
  OS                           : Linux 6.8.0-65-generic #68~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Jul 15 18:06:34 UTC 2
  Machine                      : x86_64
  R version                    : 4.5.1

= Operating System & Kernel =
$uname -a
Linux Lab-6028-214623 6.8.0-65-generic #68~22.04.1-Ubuntu SMP PREEMPT_DYNAMIC Tue Jul 15 18:06:34 UTC 2 x86_64 x86_64 x86_64 GNU/Linux
$bash -lc 'cat /etc/os-release 2>/dev/null || true'
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
$bash -lc 'lsb_release -a 2>/dev/null || true'
Distributor ID:	Ubuntu
Desc

Warning message in system(cmd, intern = TRUE, ignore.stderr = TRUE):
“running command 'update-alternatives --display libblas 2>/dev/null 2>/dev/null' had status 2”


$update-alternatives --display libblas 2>/dev/null
(no output / command not available)

= R Machine Limits & Capabilities =
  .Machine$sizeof.pointer      : 8
  .Machine$integer.max         : 2,147,483,647
  capabilities()               : jpeg, png, tiff, tcltk, X11, http/ftp, sockets, fifo, iconv, NLS, Rprof, profmem, cairo, ICU, long.double, libcurl
$ulimit -a
time(seconds)        unlimited
file(blocks)         unlimited
data(kbytes)         unlimited
stack(kbytes)        8192
coredump(blocks)     0
memory(kbytes)       unlimited
locked memory(kbytes) 32892816
process              1027576
nofiles              1048576
vmemory(kbytes)      unlimited
locks                unlimited
rtprio               0
$getconf PAGE_SIZE
4096
$getconf LONG_BIT
64

= Selected Packages =
    Package Version                                             LibPath Built
 speakeasyR   0.1.7 /home/psylab-6028/R/x86_64-pc-linux-gnu-library/4.5 4.5.1
      WGCNA    1.73 /home/psylab-6028/R/x86_64-pc-linux-gnu-libr